In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class ToyModel(nn.Module):
    r"""
    Example toy model from the original paper (page 10)

    https://arxiv.org/pdf/1703.01365.pdf


    f(x1, x2) = RELU(ReLU(x1) - 1 - ReLU(x2))
    """

    def __init__(self):
        super().__init__()

    def forward(self, input1, input2):
        relu_out1 = F.relu(input1)
        relu_out2 = F.relu(input2)
        return F.relu(relu_out1 - 1 - relu_out2)

In [5]:
from captum.attr import IntegratedGradients
model = ToyModel()

# defining model input tensors
input1 = torch.tensor([3.0], requires_grad=True)
input2 = torch.tensor([1.0], requires_grad=True)

# defining baselines for each input tensor
baseline1 = torch.tensor([0.0])
baseline2 = torch.tensor([0.0])

# defining and applying integrated gradients on ToyModel and the
ig = IntegratedGradients(model)
attributions, approximation_error = ig.attribute((input1, input2),
                                                 baselines=(baseline1, baseline2),
                                                 method='gausslegendre',
                                                 return_convergence_delta=True)

In [6]:
print(attributions, approximation_error)

(tensor([1.5000], dtype=torch.float64, grad_fn=<MulBackward0>), tensor([-0.5000], dtype=torch.float64, grad_fn=<MulBackward0>)) tensor([0.], dtype=torch.float64)


In [7]:

class Config:
    # exp settings
    name = "exp001"
    competition = "cafa-5-protein-function-prediction"
    debug = False
    seed = 8823

    train = True
    evaluation = True
    inference = True
    submission = True

    max_label = 512
    n_fold = 5
    trn_fold = [0]

    max_epochs = 8
    train_batch_size = 24
    valid_batch_size = 64
    num_workers = 4

    model = "facebook/esm2_t6_8M_UR50D"
    max_len = 512
    gradient_checkpointing = False
    gradient_accumulation_steps = 1
    clip_grad_norm = 1000

    optimizer = dict(
        optimizer_name="AdamW",
        lr=2e-5,
        weight_decay=1e-2,
        eps=1e-6,
        beta=(0.9, 0.999),
        encoder_lr=2e-5,
        decoder_lr=2e-5,
    )

    scheduler = dict(
        scheduler_name="cosine_restarts",
        first_cycle_steps_ratio=0.5,
        cycle_mult=1.0,
        max_lr=2e-5,
        min_lr=1e-7,
        warmup_steps=100,
        gamma=0.8,
    )
    batch_scheduler = True


if Config.debug:
    Config.max_label = 100
    Config.max_epochs = 2
    Config.max_len = 100
    Config.n_fold = 2
    Config.trn_fold = [0, 1]

config = Config()
print(config.scheduler)

{'scheduler_name': 'cosine_restarts', 'first_cycle_steps_ratio': 0.5, 'cycle_mult': 1.0, 'max_lr': 2e-05, 'min_lr': 1e-07, 'warmup_steps': 100, 'gamma': 0.8}


In [10]:
def get_model_config(config):
    model_config = AutoConfig.from_pretrained(config.model, output_hidden_states=True)
    model_config.hidden_dropout = 0.0
    model_config.hidden_dropout_prob = 0.0
    model_config.attention_dropout = 0.0
    model_config.attention_probs_dropout_prob = 0.0
    return model_config

model_config = get_model_config(config=config)


ValueError: Unrecognized model identifier in facebook/esm2_t6_8M_UR50D. Should contains one of 'bert', 'openai-gpt', 'gpt2', 'transfo-xl', 'xlnet', 'xlm', 'roberta', 'ctrl'

In [14]:
from esm import Alphabet

In [15]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(Config.model)

ValueError: Unrecognized model identifier in facebook/esm2_t6_8M_UR50D. Should contains one of 'bert', 'openai-gpt', 'gpt2', 'transfo-xl', 'xlnet', 'xlm', 'roberta', 'ctrl'

In [12]:
import torch
model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t6_8M_UR50D")

Using cache found in /home/andrew/.cache/torch/hub/facebookresearch_esm_main
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /home/andrew/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /home/andrew/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt


In [13]:
print(model)

ESM2(
  (embed_tokens): Embedding(33, 320, padding_idx=1)
  (layers): ModuleList(
    (0-5): 6 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=320, out_features=320, bias=True)
        (v_proj): Linear(in_features=320, out_features=320, bias=True)
        (q_proj): Linear(in_features=320, out_features=320, bias=True)
        (out_proj): Linear(in_features=320, out_features=320, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=320, out_features=1280, bias=True)
      (fc2): Linear(in_features=1280, out_features=320, bias=True)
      (final_layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=120, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((320,), eps=1e-05, elementwis

In [4]:
seq = "AAAAAPASQPAFRFGSSSTGAAAAAAPPFTFGAAPFSFGGAASGVASTPPAGSPAPPAAFTFGGSSAATGAQQPAGPVFGTPAASAPAVTFGAAPASASAAAAGGLFGAPAASSSAA"
batch_converter = alphabet.get_batch_converter()
batch_converter(["S1", seq])

KeyError: '1'

ESM2(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0): TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
    (1): TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bia